## Wie viele Aktien bruacht ein Portfolio? 

Am wie vielen Aktien lohnt sich Streeung kaum noch? 
Untersuchung mit Hilfe von Monte Carlo Simulationen

In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import PercentFormatter
import yfinance as yf

START = "2016-01-01"
END = "2026-09-30"
SEED = 42
N_SIMS = 1000
SIZES = range(1,31)
COLORS = ["#12302b", "#9fb1aa"]

In [61]:
TICKERS = {                        # ticker -> sector
    "ADS.DE": "Consumer", 
    "AIR.DE": "Industrials", 
    "ALV.DE": "Financials", 
    "BAS.DE": "Chemicals", 
    "BAYN.DE": "Healthcare", 
    "BEI.DE": "Consumer", 
    "BMW.DE": "Autos", 
    "BNR.DE": "Chemicals",
    "CBK.DE": "Financials", 
    "CON.DE": "Autos", 
    "DB1.DE": "Financials", 
    "DBK.DE": "Financials",
    "DHL.DE": "Industrials", 
    "DTE.DE": "Tech & Telecom", 
    "EOAN.DE": "Utilities & Real Estate", 
    "FME.DE": "Healthcare", 
    "FRE.DE": "Healthcare", 
    "HEI.DE": "Industrials", 
    "HEN3.DE": "Consumer", 
    "HNR1.DE": "Financials", 
    "IFX.DE": "Tech & Telecom", 
    "MBG.DE": "Autos", 
    "MRK.DE": "Healthcare",
    "MTX.DE": "Industrials", 
    "MUV2.DE": "Financials", 
    "PAH3.DE": "Autos", 
    "QIA.DE": "Healthcare",
    "RHM.DE": "Industrials", 
    "RWE.DE": "Utilities & Real Estate", 
    "SAP.DE": "Tech & Telecom",
    "SIE.DE": "Industrials", 
    "SY1.DE": "Chemicals", 
    "VNA.DE": "Utilities & Real Estate",
    "VOW3.DE": "Autos", 
    "ZAL.DE": "Consumer",
}

In [62]:
len(TICKERS)

counts ={}

for ticker, sector in TICKERS.items():
    if sector not in counts:
        counts[sector] = 0

    else:
        counts[sector] += 1

print(counts)

ticker_df = pd.Series(TICKERS)
ticker_df.value_counts()

{'Consumer': 3, 'Industrials': 5, 'Financials': 5, 'Chemicals': 2, 'Healthcare': 4, 'Autos': 4, 'Tech & Telecom': 2, 'Utilities & Real Estate': 2}


Industrials                6
Financials                 6
Healthcare                 5
Autos                      5
Consumer                   4
Chemicals                  3
Tech & Telecom             3
Utilities & Real Estate    3
Name: count, dtype: int64

## Schritt 2: Daten laden 

In [ ]:
def load_close_prices(tickers, path, start="2016-01-01", end=None):

    path=Path(path)
    if not path.exists(): 
        df = yf.download(list(tickers), start=start, auto_adjust=True, progress=False)["Close"]
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(path)
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    return df.loc[:end] if end else df


prices_raw = load_close_prices(TICKERS, "data/prices.csv", start=START, end=END)
dax_raw = load_close_prices(["^GDAXI"], "data/dax.csv", start=START, end=END)


missing = prices_raw.isna().mean().sort_values(ascending=False)

missing[missing < 0.01]

missing.head()

prices = prices_raw.copy()
dax = dax_raw.copy()

,ADS.DE,AIR.DE,ALV.DE,BAS.DE,BAYN.DE,BEI.DE,BMW.DE,BNR.DE,CBK.DE,CON.DE,...,PAH3.DE,QIA.DE,RHM.DE,RWE.DE,SAP.DE,SIE.DE,SY1.DE,VNA.DE,VOW3.DE,ZAL.DE
Date,,,,,,,,,,,,,,,,,,,,,
2016-01-04,77.898857,51.804890,91.928726,38.080582,81.662621,74.169128,51.653873,36.003563,8.241080,97.720757,...,31.737226,23.865393,50.099079,8.111809,59.454124,54.431152,53.000946,14.425573,68.500214,35.084999
2016-01-05,77.631889,52.836040,92.223373,37.866451,81.809692,74.973717,51.413113,36.225178,8.249887,98.333488,...,31.065472,24.113131,50.868938,8.283291,60.170128,54.774796,52.490223,14.522248,65.790543,33.630001
2016-01-06,76.733139,52.367332,92.518013,37.235352,80.633087,74.489120,49.710915,35.796974,8.085205,95.224396,...,30.248825,23.943117,50.430199,8.324296,60.692398,54.558430,51.997105,14.221768,64.435730,33.650002
2016-01-07,75.540703,51.259480,90.602821,36.328136,78.574074,72.669647,47.840733,34.880451,7.820132,92.909584,...,28.714317,23.263052,47.772938,8.096896,60.094303,53.514732,51.054909,13.955257,62.322174,32.990002
2016-01-08,74.259293,49.989712,89.453720,35.882980,76.404716,72.486786,46.720867,35.030693,7.646644,92.047226,...,28.888849,22.903591,48.617294,7.932871,59.850029,53.145618,50.773129,13.733166,62.376369,33.080002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-24,142.250000,194.380005,419.200012,52.299999,49.380001,78.180000,56.459999,58.860001,41.070000,69.180000,...,25.590000,38.834999,990.000000,59.160000,184.179993,274.799988,93.239998,17.090000,70.820000,22.120001
2026-09-25,146.500000,192.539993,423.500000,50.419998,50.020000,78.559998,55.720001,59.560001,42.189999,69.000000,...,25.770000,38.955002,982.400024,59.400002,185.919998,278.299988,93.080002,17.170000,71.720001,22.030001
2026-09-28,147.850006,193.179993,425.500000,50.279999,50.799999,78.559998,55.180000,59.599998,42.380001,68.760002,...,25.250000,39.305000,966.299988,59.139999,184.020004,278.450012,93.379997,17.055000,70.879997,21.799999


In [ ]:
#compare if days of prices and dax are identically 

days_price = set(prices)
days_dax = set(dax)


-4


^GDAXI    0.0
dtype: float64